# Step 15 — Asociación estadística El Niño–amenaza

Notebook delgado: toda la lógica vive en `src/analisis/asociacion_enso.py`. Documentación: `docs/methodology/15_enso_hazard_association.md` y `docs/executive/15_enso_hazard_association.md`.

**English:** ENSO-Hazard Association

Runs the statistical association between El Nino and hazard events (negative binomial with seasonality control) through the `src.analisis.asociacion_enso` module and exports the results to `data/processed/` and `outputs/`.

In [ ]:
# System path handling, logging and warnings
import sys, logging, warnings  # utilidades estándar: ruta, logging y avisos
# YAML reader for the configuration
import yaml  # lectura de la configuración
# Tabular data handling
import pandas as pd  # lectura de las tablas procesadas
# Make the project's src package importable
sys.path.insert(0, '..')  # permite importar el paquete src desde notebooks/
# Statistical association module
from src.analisis import asociacion_enso as ae  # módulo con la lógica de la fase
# Show INFO-level progress messages
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')  # logging, no print
# Hide warnings in the notebook output
warnings.filterwarnings('ignore')  # silencia avisos de convergencia no críticos de statsmodels

In [ ]:
# Load the association settings from the config
cfg = yaml.safe_load(open('../config/config.yaml', encoding='utf-8'))['enso_association']  # parámetros de la fase
# Load the hazard events
eventos = pd.read_parquet('../data/processed/hazard_event.parquet')  # 21,743 eventos de la Fase 1/2
# Load the extended monthly ENSO series
enso = pd.read_parquet('../data/processed/enso_index_monthly_extended.parquet')  # índice ENSO mensual (132 meses)

In [ ]:
# Run the full analysis on every region and hazard stratum
resultados = ae.run_analysis(eventos, enso, cfg)  # corre todos los estratos y especificaciones (~1 min)
# Export the result tables
ae.export_tables(resultados, '../' + cfg['salida_dir'])  # escribe Parquet + CSV en data/processed

In [ ]:
# Results of the incidence rate ratios
irr = resultados['irr']  # tabla de razones de tasas de todos los modelos
# Show the primary model: event-rate ratio during El Nino, with its confidence interval
irr[irr['modelo'] == 'M1_primario_ICEN_binario'][['estrato', 'n_eventos', 'IRR_crudo', 'IRR', 'IC95_inf', 'IC95_sup', 'p_BH']].round(3)  # modelo primario